# RQ3 — Adaptation continue (version réduite), pli 0

`adaptation/continual.py` sur le `best.pt` de la baseline fp32 (notebook `rq3-baseline-fp32-pli0`, terminé) : les 6 centres OOD
présentés un par un (24 sujets par centre, lots de 4), évaluation préquentielle (chaque lot est noté avant de servir à
l'adaptation), 4 stratégies (modèle figé, auto-apprentissage naïf, TENT seul, adaptation contrôlée = TENT +
champion/challenger + replay buffer), Dice sur les anciens centres pour mesurer l'oubli, détection du décalage par distance
de Wasserstein entre histogrammes d'intensité (`center_distance()`, analysis/). Durée estimée sur une T4 : 30 à 50 min.

**Kaggle** : *GPU T4*, Internet activé, datasets `atlas-c1-processed` et `projet-avc-code`, sortie du notebook `rq3-baseline-fp32-pli0` en entrée. Sorties dans `/kaggle/working/adaptation_continue/`.

In [ ]:
# Environnement : GPU obligatoire, sinon arrêt immédiat avec un message clair ; écarts à Kaggle signalés.
import hashlib, json, os, re, shutil, subprocess, sys, time, zipfile
from pathlib import Path
import torch
PLATFORM = os.environ.get("RQ3_PLATFORM", "kaggle")
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "aucun"
print("Plateforme", PLATFORM, "| torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPU", GPU,
      "| Python", sys.version.split()[0])
if not torch.cuda.is_available() and os.environ.get("RQ3_GPU", "1") == "1":
    where = ("Exécution > Modifier le type d'exécution > GPU T4" if PLATFORM == "colab"
             else "Settings > Accelerator > « GPU T4 x2 »")
    raise RuntimeError(f"AUCUN GPU DISPONIBLE : {where}, puis relancer. Rien n'a été lancé.")
INPUT = Path(os.environ.get("RQ3_INPUT", "/kaggle/input"))      # datasets
WORKING = Path(os.environ.get("RQ3_WORKING", "/kaggle/working"))  # code
RUNS_DIR = Path(os.environ.get("RQ3_RUNS", str(WORKING / "runs")))  # toutes les sorties des runs
SCRATCH = Path(os.environ.get("RQ3_TMP", "/tmp/rq3"))            # liens vers les données, jamais dans les sorties
EXPECTED_COMMIT = None  # commit attendu dans VERSION.json du dataset de code (None : pas de contrôle)
REFERENCE = {'torch': '2.11.0+cu130', 'gpu': 'Tesla T4'}  # environnement Kaggle de la baseline
differences = []
if REFERENCE["gpu"] not in GPU:
    differences.append(f"GPU {GPU} au lieu de {REFERENCE['gpu']} : durées et arrondis différents de Kaggle")
if torch.__version__ != REFERENCE["torch"]:
    differences.append(f"torch {torch.__version__} au lieu de {REFERENCE['torch']} (Kaggle)")
for difference in differences:
    print("AVERTISSEMENT (la run continue) :", difference)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
(RUNS_DIR / f"environnement_{PLATFORM}_{time.strftime('%Y%m%d_%H%M%S')}.json").write_text(json.dumps(dict(
    plateforme=PLATFORM, torch=torch.__version__, cuda=torch.version.cuda, gpu=GPU, python=sys.version.split()[0],
    reference=REFERENCE, avertissements=differences), indent=2, ensure_ascii=False))

In [ ]:
# Code du projet : dataset privé projet-avc-code, cherché récursivement dans le dossier d'entrée.
def code_version(candidate):
    if candidate.suffix == ".zip":
        with zipfile.ZipFile(candidate) as archive:
            return json.loads(archive.read("VERSION.json")) if "VERSION.json" in archive.namelist() else None
    return json.loads(candidate.read_text())

candidates = sorted(INPUT.rglob("projet_avc_code.zip")) + sorted(
    path for path in INPUT.rglob("VERSION.json") if (path.parent / "adaptation" / "train.py").is_file())
found = [(path, code_version(path)) for path in candidates]
found = [(path, version) for path, version in found if version]
if not found:
    raise FileNotFoundError("Dataset projet-avc-code introuvable : l'ajouter en entrée, puis relancer.")
matching = [(path, version) for path, version in found if EXPECTED_COMMIT in (None, version["commit"])]
if not matching:
    raise RuntimeError(f"VERSION.json : commit {sorted({v['commit'] for _, v in found})} au lieu de {EXPECTED_COMMIT}. "
                       "Mettre l'entrée projet-avc-code à sa dernière version, puis relancer.")
source, VERSION = matching[0]
print("Code :", source, VERSION)
REPO = WORKING / "PROJET_AVC"
if not REPO.exists():
    if source.suffix == ".zip":
        zipfile.ZipFile(source).extractall(REPO)
    else:  # archive décompressée : copie de l'arborescence
        shutil.copytree(source.parent, REPO)
if os.environ.get("RQ3_PIP", "1") == "1":
    pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "monai==1.6.1", "torchio==1.2.1",
                          "nibabel==5.4.2", "pyarrow"])
    if pip.returncode:
        raise RuntimeError("Installation de MONAI / TorchIO impossible : activer Internet, puis relancer.")

In [ ]:
# Données C1 : dataset privé atlas-c1-processed, cherché récursivement ; liens vers les dossiers de sujets.
tables = sorted(INPUT.rglob("subjects.parquet"))
if len(tables) != 1:
    raise FileNotFoundError(f"subjects.parquet trouvé {len(tables)} fois (1 attendu) : "
                            "ajouter le dataset atlas-c1-processed en entrée.")
TABLE = tables[0]
DATA = SCRATCH / "processed"
DATA.mkdir(parents=True, exist_ok=True)
subjects = sorted(INPUT.rglob("qc.json"))
if not subjects:  # archives restées compressées : extraction hors des sorties
    for archive in sorted(INPUT.rglob("processed_partie*.zip")):
        zipfile.ZipFile(archive).extractall(SCRATCH / "c1_extrait")
    subjects = sorted((SCRATCH / "c1_extrait").rglob("qc.json"))
for qc in subjects:
    link = DATA / qc.parent.name
    if not link.exists():
        link.symlink_to(qc.parent, target_is_directory=True)
if not (DATA / "subjects.parquet").exists():
    (DATA / "subjects.parquet").symlink_to(TABLE)
print(len(subjects), "sujets prétraités ; table", TABLE)
if not subjects:
    raise FileNotFoundError("Aucun dossier de sujet (qc.json) dans atlas-c1-processed")
# Même découpage que le code : splits.json du dataset (à côté de subjects.parquet) identique à celui du dépôt.
data_splits = next((folder / "splits.json" for folder in [TABLE.parent, *TABLE.parents]
                    if folder != INPUT and INPUT in folder.parents and (folder / "splits.json").is_file()), None)
if data_splits is None:
    raise FileNotFoundError("splits.json absent du dataset atlas-c1-processed")
digest = lambda path: hashlib.sha256(Path(path).read_bytes()).hexdigest()
assert digest(data_splits) == digest(REPO / "splits.json"), "splits.json du dataset différent de celui du code"

In [ ]:
# Checkpoint best.pt de la baseline fp32 (sortie du notebook rq3-baseline-fp32-pli0).
checkpoints = [path for path in INPUT.rglob("best.pt") if path.parent.name == "unet3d_baseline_pli0"]
if not checkpoints:
    raise FileNotFoundError("best.pt de la baseline introuvable : le notebook rq3-baseline-fp32-pli0 doit être terminé et "
                            "attaché en entrée (Kaggle : Add Input > Notebooks).")
CHECKPOINT = checkpoints[0]
state = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
assert all(torch.isfinite(value).all() for value in state["model"].values() if value.is_floating_point()), \
    "best.pt contient des poids non finis"
print("Checkpoint :", CHECKPOINT, state["state"])
OUTPUT = RUNS_DIR.parent / "adaptation_continue"

In [ ]:
# Flux des centres OOD, évaluation préquentielle, 4 stratégies, oubli (journal aussi dans adaptation_continue/).
# Courte (30 à 50 min) : après une coupure, elle recommence du début (sorties réécrites).
started = time.time()
command = [sys.executable, "-u", "-m", "adaptation.continual", "--checkpoint", str(CHECKPOINT), "--data-root", str(DATA),
           "--output", str(OUTPUT), "--subjects-per-center", "24", "--lot-size", "4"]
OUTPUT.mkdir(parents=True, exist_ok=True)
with open(OUTPUT / "adaptation_continue.log", "w", encoding="utf-8") as log:
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
    process.wait()
print(f"{(time.time() - started) / 60:.1f} min")
if process.returncode:
    raise RuntimeError(f"adaptation.continual terminé avec le code {process.returncode}")
for path in sorted(OUTPUT.iterdir()):
    print(f"{path.stat().st_size / 1e3:10.1f} ko  {path.name}")